# Consumo Final de Energía por Energético — BECO 2006-2024

Lee el valor de **CONSUMO FINAL** directamente desde la hoja `BECO_Energéticos 2006-2024`
iterando cada energético con xlwings (Excel calcula, Python lee el resultado).

**Requisito:** descargar `BECO_VER_01_2024_1975_2024.xlsx` desde la página del BECO de la UPME
(`upme.gov.co/simec/oferta-y-demanda/balance-energetico-colombiano/`) y ubicarlo en esta carpeta.
Requiere Excel instalado (xlwings automatiza Excel de escritorio).

In [ ]:
import xlwings as xw
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

ARCHIVO = Path("BECO_VER_01_2024_1975_2024.xlsx").resolve()
AÑOS    = list(range(2006, 2025))

COLORES = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990",
    "#dcbeff", "#9a6324", "#c0c0c0", "#800000", "#aaffc3",
    "#808000", "#ffd8b1", "#000075", "#a9a9a9", "#4682b4",
]

In [ ]:
def extraer_consumo_final() -> pd.DataFrame:
    """
    Abre el archivo en Excel, itera cada energético del selector (celda E5),
    y lee la fila CONSUMO FINAL (fila 41, columnas D-V = años 2006-2024)
    ya calculada por Excel.
    """
    app = xw.App(visible=False, add_book=False)
    try:
        wb  = app.books.open(str(ARCHIVO))
        ws  = wb.sheets["BECO_Energéticos 2006-2024"]

        años_hoja = ws.range("D7:V7").value
        selector_original = ws.range("E5").value

        # Leer códigos y nombres desde BECO_Fuentes
        ws_f    = wb.sheets["BECO_Fuentes"]
        codigos = [c for c in ws_f.range("C4:V4").value if c and c != "Balance energético Colombiano\nFuentes de información"]
        nombres = ws_f.range("C5:V5").value
        cod_a_nombre = {str(codigos[i]): str(nombres[i]) for i in range(len(codigos)) if codigos[i] and nombres[i]}

        data = {}
        for codigo, nombre in cod_a_nombre.items():
            ws.range("E5").value = codigo
            wb.app.calculate()
            valores = ws.range("D41:V41").value
            data[nombre] = {int(años_hoja[i]): (v or 0.0) for i, v in enumerate(valores)}
            print(f"  {codigo} ({nombre}): OK")

        # Restaurar selector y cerrar sin guardar
        ws.range("E5").value = selector_original
        wb.close()
        return pd.DataFrame(data).T.sort_index(axis=1)
    finally:
        app.quit()

In [ ]:
def graficar(df: pd.DataFrame) -> None:
    """Gráfica interactiva con botones para cambiar entre área apilada y líneas."""
    df = df.loc[(df > 0).any(axis=1)].fillna(0)
    anos = df.columns.tolist()
    energeticos = df.index.tolist()

    trazas_proxy    = []  # leyenda: cuadrado de color
    trazas_area     = []  # área apilada visual
    trazas_barhover = []  # go.Bar width=0 -> invisible en gráfico, swatch cuadrado en tooltip
    trazas_linea    = []  # modo líneas

    for i, nombre in enumerate(energeticos):
        color   = COLORES[i % len(COLORES)]
        valores = df.loc[nombre].tolist()
        tooltip = f"<b>{nombre}</b>: %{{y:,.2f}} PJ<extra></extra>"

        trazas_proxy.append(go.Scatter(
            x=[None], y=[None], name=nombre,
            mode="markers",
            marker=dict(symbol="square", size=10, color=color, line=dict(width=0)),
            legendgroup=nombre,
            showlegend=True,
        ))
        trazas_area.append(go.Scatter(
            x=anos, y=valores, name=nombre,
            mode="lines", stackgroup="one",
            fillcolor=color, line=dict(color=color, width=0.5),
            legendgroup=nombre,
            showlegend=False,
            hoverinfo="skip",
        ))
        # Barras de ancho cero: invisibles en el gráfico, swatch cuadrado en tooltip unificado
        trazas_barhover.append(go.Bar(
            x=anos, y=valores, name=nombre,
            marker=dict(color=color, opacity=0),
            width=0,
            legendgroup=nombre,
            showlegend=False,
            hovertemplate=tooltip,
        ))
        trazas_linea.append(go.Scatter(
            x=anos, y=valores, name=nombre,
            mode="lines+markers",
            line=dict(color=color, width=2),
            marker=dict(symbol="square", size=6, color=color, line=dict(width=0)),
            legendgroup=nombre,
            showlegend=False,
            visible=False,
            hovertemplate=tooltip,
        ))

    n = len(energeticos)
    vis_area  = [True] * n + [True]  * n + [True]  * n + [False] * n
    vis_linea = [True] * n + [False] * n + [False] * n + [True]  * n

    fig = go.Figure(data=trazas_proxy + trazas_area + trazas_barhover + trazas_linea)
    fig.update_layout(
        title=dict(
            text="Consumo Final de Energía por Energético — Colombia 2006-2024",
            font=dict(size=16), x=0.5,
        ),
        width=950, height=560,
        xaxis=dict(title="Año", tickmode="linear", dtick=1, tickangle=-45),
        yaxis=dict(title="Petajulios (PJ)", tickformat=",.0f", gridcolor="#e0e0e0"),
        legend=dict(
            title="Energético", orientation="v",
            x=1.01, y=0.97,
            xanchor="left", yanchor="top",
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#cccccc", borderwidth=1,
            font=dict(size=11),
            itemwidth=30,
            tracegroupgap=0,
        ),
        hovermode="x unified",
        hoverlabel=dict(font=dict(size=10), namelength=0),
        plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(l=70, r=220, t=80, b=80),
        updatemenus=[dict(
            type="buttons", direction="left",
            x=0.0, y=1.07, xanchor="left", showactive=True,
            buttons=[
                dict(label="Área apilada", method="update", args=[{"visible": vis_area}]),
                dict(label="Líneas",       method="update", args=[{"visible": vis_linea}]),
            ],
        )],
    )
    fig.show()

In [ ]:
print("Leyendo valores directamente desde Excel (puede tardar ~30 segundos)...")
df = extraer_consumo_final()

df = df / 1000  # TJ → PJ

salida_csv = ARCHIVO.parent / "consumo_final_energeticos.csv"
df.to_csv(salida_csv, encoding="utf-8-sig")
print(f"\nTabla exportada en: {salida_csv}")
print()
df.round(3)

In [ ]:
graficar(df)